# B4 — Navigation Styles

Classifies trajectory navigation patterns for the museum context. A trajectory can have multiple styles simultaneously.

- **CasualMovement**: low artwork count + short avg dwell (quick walk-through)
- **ExploratoryMovement**: high artwork count (seeing everything)
- **ImmersiveMovement**: high avg dwell (deep engagement)
- **TargetedSegment**: few artworks + significant dwell (specific artworks only)

Thresholds derived from BIRD's `semantic_info_entire_trajectories.csv`.

In [ ]:
import os
import sys
import pandas as pd

sys.path.insert(0, os.path.abspath('helpers'))
from fuseki_config import query_endpoint, update_endpoint, PROJECT_ROOT
from fuseki_queries import execute_sparql_query, execute_sparql_update, format_results

In [ ]:
PREFIXES = """
PREFIX trajecto: <http://w3id.org/trajecto#>
PREFIX poi:      <http://w3id.org/trajecto/poi#>
PREFIX patterns: <http://w3id.org/trajecto/patterns#>
PREFIX dul:      <http://www.ontologydesignpatterns.org/ont/dul/DUL.owl#>
PREFIX xsd:      <http://www.w3.org/2001/XMLSchema#>
"""

## 1. Load Semantic Info for Threshold Calibration

In [ ]:
sem_info_path = PROJECT_ROOT / 'data' / 'bird' / 'bird-dataset-main' / 'data' / 'semantic_info_entire_trajectories.csv'
sem_info = pd.read_csv(sem_info_path)
print(f'Trajectories in semantic info: {len(sem_info)}')
print('\nDistributions:')
for col in ['duration', 'speed', 'nb_items', 'nb_stops']:
    print(f'  {col}: mean={sem_info[col].mean():.1f}, median={sem_info[col].median():.1f}, '
          f'std={sem_info[col].std():.1f}')

## 2. Compute Trajectory Statistics from KG

In [ ]:
# Get per-trajectory statistics from the KG
query = PREFIXES + """
SELECT ?traj
       (COUNT(DISTINCT ?poi) AS ?artworkCount)
       (AVG(?dwell) AS ?avgDwell)
       (MAX(?dwell) AS ?maxDwell)
       (COUNT(?interaction) AS ?totalInteractions)
WHERE {
    ?pos dul:isPartOf ?traj .
    ?pos trajecto:hasInteraction ?interaction .
    ?interaction poi:interactsWith ?poi .
    ?interaction poi:hasDwellTimeInSeconds ?dwell .
}
GROUP BY ?traj
"""
result = execute_sparql_query(query_endpoint, query)
stats = format_results(result)
stats['artworkCount'] = stats['artworkCount'].astype(int)
stats['avgDwell'] = stats['avgDwell'].astype(float)
stats['maxDwell'] = stats['maxDwell'].astype(float)
stats['totalInteractions'] = stats['totalInteractions'].astype(int)

print(f'Trajectories with interactions: {len(stats)}')
stats.describe()

## 3. Classify Navigation Styles

In [ ]:
# Thresholds based on distribution analysis
median_count = stats['artworkCount'].median()
median_dwell = stats['avgDwell'].median()

def classify_navigation(row):
    count = row['artworkCount']
    avg_dwell = row['avgDwell']
    styles = []

    # Casual: low artwork count + short dwell
    if count < median_count * 0.5 and avg_dwell < median_dwell:
        styles.append('CasualMovement')

    # Exploratory: high artwork count
    if count > median_count * 1.2:
        styles.append('ExploratoryMovement')

    # Immersive: high avg dwell
    if avg_dwell > median_dwell * 1.5:
        styles.append('ImmersiveMovement')

    # Targeted: few artworks + significant dwell
    if count < median_count * 0.5 and avg_dwell >= median_dwell:
        styles.append('TargetedSegment')

    # Steady: moderate count AND moderate dwell (covers the middle zone)
    if (count >= median_count * 0.5 and count <= median_count * 1.2
            and avg_dwell >= median_dwell * 0.8 and avg_dwell <= median_dwell * 1.5):
        styles.append('SteadyMovement')

    return styles

stats['navStyles'] = stats.apply(classify_navigation, axis=1)

# Summary
from collections import Counter
style_counts = Counter()
for styles in stats['navStyles']:
    for s in styles:
        style_counts[s] += 1

print(f'Median artwork count: {median_count}, median dwell: {median_dwell:.1f}s')
print('\nNavigation style distribution:')
for style, count in style_counts.most_common():
    print(f'  {style}: {count}')

no_style = (stats['navStyles'].apply(len) == 0).sum()
multi_style = (stats['navStyles'].apply(len) > 1).sum()
print(f'\nTrajectories with no style: {no_style}')
print(f'Trajectories with multiple styles: {multi_style}')

## 4. Insert Navigation Patterns into KG

In [ ]:
insert_lines = []
for _, row in stats.iterrows():
    for style in row['navStyles']:
        insert_lines.append(
            f'  <{row["traj"]}> patterns:exhibitsPattern patterns:{style} .'
        )

if insert_lines:
    triples = '\n'.join(insert_lines)
    update = PREFIXES + f"""INSERT DATA {{\n{triples}\n}}"""
    success = execute_sparql_update(update_endpoint, update)
    print(f'Inserted {len(insert_lines)} navigation patterns: {"OK" if success else "FAILED"}')

## 5. Verify

In [ ]:
q = PREFIXES + """
SELECT ?style (COUNT(?traj) AS ?count)
WHERE {
    ?traj patterns:exhibitsPattern ?style .
}
GROUP BY ?style
ORDER BY DESC(?count)
"""
result = execute_sparql_query(query_endpoint, q)
format_results(result)